# Appendix C.2 — Water Quality Archive (observations and compliance samples)

Evidence for the claims made about the EA Water Quality Archive in Appendix C.

**Sources used**

| file | what it is |
| --- | --- |
| `ttl/breaches/compliance_observations.csv` | compliance samples fetched with `complianceOnly=true` (committed cache, 2000–2026) |
| `raw_datasets/poole_harbour_rivers_water_quality_observations_2020_2026_combined.csv` | the bulk catchment download, 2020–2026 |
| `ttl/regulation/sampling_points.csv` | sampling-point reference facts resolved from the archive |
| `ttl/regulation.ttl` | the delivered graph, for what is and is not linked |

The last section makes a live HTTP request to `environment.data.gov.uk`; it degrades gracefully offline.


In [1]:
import os, json, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "raw_datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
RAW = ROOT / "raw_datasets"
REG = RAW / "access_database_csv_files"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
print("repository root:", ROOT)


repository root: /Users/waf/git/projects/demonstrator-poc


In [2]:
comp = pd.read_csv(ROOT / "ttl" / "breaches" / "compliance_observations.csv", dtype=str, low_memory=False)
comp["when"] = pd.to_datetime(comp.phenomenonTime)
bulk = pd.read_csv(RAW / "poole_harbour_rivers_water_quality_observations_2020_2026_combined.csv",
                   dtype=str, low_memory=False)
print(f"compliance samples : {len(comp):,} rows, {comp.when.dt.year.min()}-{comp.when.dt.year.max()}")
print(f"bulk download      : {len(bulk):,} rows")
comp.head(3)


compliance samples : 44,999 rows, 2000-2026
bulk download      : 54,481 rows


,id,samplingPoint.notation,phenomenonTime,samplingPurpose,sampleMaterialType,determinand.notation,result,unit,when
0,http://environment.data.gov.uk/water-quality/sampling-point/SW-50410119/sample/2882195...,SW-50410119,2006-09-11T09:54:00,COMPLIANCE AUDIT (PERMIT),FINAL SEWAGE EFFLUENT,0061,7.97,PH UNITS,2006-09-11 09:54:00
1,http://environment.data.gov.uk/water-quality/sampling-point/SW-50410119/sample/2885599...,SW-50410119,2006-10-09T11:47:00,COMPLIANCE AUDIT (PERMIT),FINAL SEWAGE EFFLUENT,0061,8.08,PH UNITS,2006-10-09 11:47:00
2,http://environment.data.gov.uk/water-quality/sampling-point/SW-50410119/sample/2888564...,SW-50410119,2006-11-08T11:45:00,COMPLIANCE AUDIT (PERMIT),FINAL SEWAGE EFFLUENT,0061,7.93,PH UNITS,2006-11-08 11:45:00


---
## C.2.1 Sampling point linkage — 91 of 167 points have no permit association

> *"91 of the 167 sampling points in the catchment have no permit association. What would an adverse
> observational result be in these situations?"*


In [3]:
import pyoxigraph as ox

store = ox.Store()
store.bulk_load(path=str(ROOT / "ttl" / "regulation.ttl"), format=ox.RdfFormat.TURTLE)
P = """PREFIX water: <http://environment.data.gov.uk/ontology/water/>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#> """

def sparql(q, cols):
    return pd.DataFrame([[None if v is None else v.value for v in r] for r in store.query(P + q)],
                        columns=cols)

points = sparql("SELECT DISTINCT ?sp WHERE { ?sp a sosa:FeatureOfInterest }", ["sp"])
linked = sparql("SELECT DISTINCT ?sp WHERE { ?dp water:monitoredAt ?sp }", ["sp"])
print(f"sampling points in the catchment          : {len(points)}")
print(f"named by a permit as its discharge monitor: {len(linked)}")
print(f"with NO permit association                : {len(points) - len(linked)}")


sampling points in the catchment          : 167
named by a permit as its discharge monitor: 76
with NO permit association                : 91


In [4]:
sp_ref = pd.read_csv(ROOT / "ttl" / "regulation" / "sampling_points.csv", dtype=str)
unlinked = sp_ref[~sp_ref.sp_notation.isin(linked.sp.str.rsplit("/", n=1).str[-1])]
print("What the unlinked points actually are:")
print(unlinked.type_label.value_counts().to_string())


What the unlinked points actually are:
type_label
FRESHWATER - RIVERS                                          48
GROUNDWATER - BOREHOLE                                       18
POLLUTION/INVESTIGATION POINTS - ENVIRONMENT                 14
FRESHWATER - COMPARATIVE INLET POINTS                         4
SALINE WATER - DESIGNATED BATHING BEACHES                     2
FRESHWATER - LAKES/PONDS/RESERVOIRS                           2
TRADE DISCHARGES - PROCESS EFFLUENT - WATER COMPANY (WTW)     1
GROUNDWATER - SPRING                                          1
SALINE WATER - ESTUARINE SITES - NON BATHING/SHELLFISH        1


In [5]:
print("A sample of them, with their observation counts in the bulk download:")
counts = bulk.groupby("samplingPoint.notation").size().rename("observations_2020_26")
view = unlinked.merge(counts, left_on="sp_notation", right_index=True, how="left")
view[["sp_notation","pref_label","type_label","observations_2020_26"]].sort_values(
    "observations_2020_26", ascending=False).head(10)


A sample of them, with their observation counts in the bulk download:


,sp_notation,pref_label,type_label,observations_2020_26
98,SW-50590127,RIVER FROME AT HOLME BRIDGE,FRESHWATER - RIVERS,7950
33,SW-50450129,RIVER PIDDLE AT WEST MILLS WAREHAM,FRESHWATER - RIVERS,3212
143,SW-RSN0029,RSN0029 AT FROME VAUGHCHURCH HOUSE,FRESHWATER - RIVERS,2372
70,SW-5056GW07,CHURCH FARM BOREHOLE,GROUNDWATER - BOREHOLE,1442
150,SW-RSN1389,RSN1389 MILL HOUSE,FRESHWATER - RIVERS,1374
112,SW-50950270,POOLE HARBOUR AT LAKE (19400),SALINE WATER - DESIGNATED BATHING BEACHES,1261
113,SW-50950300,POOLE HARBOUR ROCKLEY SANDS (19450),SALINE WATER - DESIGNATED BATHING BEACHES,1221
69,SW-5056GW05,CAME DOWN GOLF CLUB BOREHOLE,GROUNDWATER - BOREHOLE,1111
66,SW-5055GW07,MOUNT PLEASANT FARM BOREHOLE CERNE ABBAS,GROUNDWATER - BOREHOLE,1000
149,SW-RSN1197,RSN1197 RIVER HOOKE D/S BRIDGE FARM,FRESHWATER - RIVERS,959


These are rivers, boreholes, bathing waters and investigation points. A result at one of them cannot be
adverse *against anything*: no permit names them, so there is no limit to test and no party the result
attaches to. The archive publishes the measurement; nothing published says what it means.


---
## C.2.2 Non-detects are recorded as text, and dropping them manufactures failures

> *"36.9% of compliance results are `<` non-detects — 66.1% of BOD, 38.0% of suspended solids, 35.8% of
> ammonia … Dropping them both inflates means and shrinks the sample count, which tightens the percentile
> look-up band and manufactures failures that did not occur."*

The percentages below are recomputed from the committed compliance cache (44,999 observations) each time
this notebook runs.


In [6]:
comp["non_detect"] = comp.result.str.startswith("<")
print(f"non-detect share of ALL compliance results: {100 * comp.non_detect.mean():.1f}%\n")

names = {"0085": "BOD : 5 Day ATU", "0135": "Solids, Suspended", "0111": "Ammoniacal Nitrogen as N",
         "0348": "Phosphorus, Total as P"}
rows = []
for code, label in names.items():
    s = comp[comp["determinand.notation"] == code]
    rows.append({"determinand": f"{code} {label}", "results": len(s),
                 "non_detects": int(s.non_detect.sum()),
                 "share_%": round(100 * s.non_detect.mean(), 1)})
pd.DataFrame(rows)


non-detect share of ALL compliance results: 36.9%



,determinand,results,non_detects,share_%
0,0085 BOD : 5 Day ATU,8172,5399,66.1
1,"0135 Solids, Suspended",10586,4026,38.0
2,0111 Ammoniacal Nitrogen as N,6485,2321,35.8
3,"0348 Phosphorus, Total as P",788,1,0.1


In [7]:
print("The actual records -- these are measurements, not missing data:")
comp[comp.non_detect][["samplingPoint.notation","phenomenonTime","determinand.notation","result","unit"]].head(6)


The actual records -- these are measurements, not missing data:


,samplingPoint.notation,phenomenonTime,determinand.notation,result,unit
332,SW-50410119,2009-02-24T11:25:00,0085,<4.62,MILLIGRAM PER LITRE
334,SW-50410119,2009-04-22T12:40:00,0085,<6,MILLIGRAM PER LITRE
337,SW-50410119,2009-07-03T06:25:00,0085,<9,MILLIGRAM PER LITRE
339,SW-50410119,2009-09-24T12:00:00,0085,<9,MILLIGRAM PER LITRE
341,SW-50410119,2009-12-21T11:30:00,0085,<11,MILLIGRAM PER LITRE
348,SW-50410119,2010-07-15T12:15:00,0085,<6,MILLIGRAM PER LITRE


In [8]:
print("Other non-numeric forms in the same column:")
print(comp[comp.result.str.startswith(">")].result.value_counts().head(5).to_string(), "\n")
free_text = comp[~comp.result.str.match(r"^[<>]?\s*-?\d+(\.\d+)?$", na=False)]
print("free text (not a measurement at all):")
print(free_text.result.value_counts().to_string())


Other non-numeric forms in the same column:
result
>33     2
>28     2
>23     1
>122    1
>30     1 

free text (not a measurement at all):
result
Not found    77


### The manufactured failure, computed

Poole WRC (sampling point `SW-50950709`), BOD, 95th-percentile limit 20 mg/l. The EA assesses this by
counting exceedances over a 12-month sample set and comparing against the maximum its look-up table
allows **for that number of samples** — so the sample count is part of the test.


In [9]:
# The EA's look-up table, as used by ttl/breaches/breaches_to_db.py
LUT = [(4, 7, 1), (8, 16, 2), (17, 28, 3), (29, 40, 4), (41, 53, 5), (54, 67, 6), (68, 81, 7)]
def allowed(n):
    for lo, hi, k in LUT:
        if lo <= n <= hi:
            return k
    return None

poole = comp[(comp["samplingPoint.notation"] == "SW-50950709") & (comp["determinand.notation"] == "0085")]
out = []
for year in [2021, 2022, 2023, 2024]:
    y = poole[poole.when.dt.year == year]
    kept = y[~y.non_detect]
    def values(frame, non_detects_as_zero):
        v = frame.result.str.lstrip("<>").astype(float)
        if non_detects_as_zero:
            v = v.where(~frame.non_detect, 0.0)     # the EA's rule: '<5' IS a result, and it is zero
        return v

    for label, frame, as_zero in [("EA rule: non-detects counted as zero", y, True),
                                  ("non-detects dropped", kept, False)]:
        v = values(frame, as_zero)
        n, exceed = len(frame), int((v > 20.0).sum())
        out.append({"year": year, "treatment": label, "samples": n, "exceedances": exceed,
                    "LUT allows": allowed(n),
                    "verdict": "FAIL" if allowed(n) is not None and exceed > allowed(n) else "pass"})
pd.DataFrame(out)


,year,treatment,samples,exceedances,LUT allows,verdict
0,2021,EA rule: non-detects counted as zero,36,0,4,pass
1,2021,non-detects dropped,5,0,1,pass
2,2022,EA rule: non-detects counted as zero,36,0,4,pass
3,2022,non-detects dropped,7,0,1,pass
4,2023,EA rule: non-detects counted as zero,36,3,4,pass
5,2023,non-detects dropped,15,3,2,FAIL
6,2024,EA rule: non-detects counted as zero,36,0,4,pass
7,2024,non-detects dropped,12,0,2,pass


In 2023 the same 3 exceedances are a **pass** on the EA's rule (36 samples, 4 allowed) and a **FAIL**
once non-detects are dropped (15 samples, 2 allowed). The breach is an artefact of the coercion, not an
event at the works. Note the direction: the discarded values are the *low* ones, so their loss is not
random.


---
## C.2.3 "No discharge" is recorded as a coded result, not as a flag on the sample

> *"Samples taken when nothing was being discharged are excluded by the guidance; the archive records
> this only as a coded result against a dedicated determinand."*


In [10]:
no_flow = bulk[bulk["determinand.prefLabel"].astype(str).str.contains("No flow", case=False, na=False)]
print(f'rows in the bulk download recording "no flow/discharge": {len(no_flow):,}\n')
no_flow[["samplingPoint.notation","samplingPoint.prefLabel","phenomenonTime",
         "determinand.notation","determinand.prefLabel","result","unit"]].head(5)


rows in the bulk download recording "no flow/discharge": 1,015



,samplingPoint.notation,samplingPoint.prefLabel,phenomenonTime,determinand.notation,determinand.prefLabel,result,unit
40,SW-50410119,PUDDLETOWN STW FE,2022-10-07 08:05:00,7668,No flow /No sample,No flow/discharge at sampling point,Coded Result
556,SW-50430126,BROCKHILL WATERCRESS FARM B & C 2,2022-01-11 10:55:00,7668,No flow /No sample,No flow/discharge at sampling point,Coded Result
557,SW-50430126,BROCKHILL WATERCRESS FARM B & C 2,2022-02-02 10:36:00,7668,No flow /No sample,No flow/discharge at sampling point,Coded Result
558,SW-50430126,BROCKHILL WATERCRESS FARM B & C 2,2022-03-09 10:57:00,7668,No flow /No sample,No flow/discharge at sampling point,Coded Result
559,SW-50430126,BROCKHILL WATERCRESS FARM B & C 2,2022-04-08 14:34:00,7668,No flow /No sample,No flow/discharge at sampling point,Coded Result


It is determinand `7668`, with the coded result `No flow /No sample` — an *observation of its own*,
not an attribute of the sample. Two consequences:

1. A consumer filtering the results column to numerics drops it, and with it the fact that there was
   nothing to sample. "No discharge" and "no data" become indistinguishable.
2. The exclusion cannot be applied to the case that matters — flow nil but a number still returned —
   because no flow record accompanies the sample:


In [11]:
print("Is determinand 7668 present in the compliance-sample set at all?")
print((comp["determinand.notation"] == "7668").sum(), "rows\n")
print("Columns available on a compliance sample:")
print([c for c in comp.columns if c != "when"])
print("\nNothing here carries discharge flow, so 'nil flow, numeric result' cannot be detected.")


Is determinand 7668 present in the compliance-sample set at all?
0 rows

Columns available on a compliance sample:
['id', 'samplingPoint.notation', 'phenomenonTime', 'samplingPurpose', 'sampleMaterialType', 'determinand.notation', 'result', 'unit', 'non_detect']

Nothing here carries discharge flow, so 'nil flow, numeric result' cannot be detected.


---
## C.2.4 A granted waiver does not change what the archive says the sample *is*

> *"`complianceOnly=true` filters on `samplingPurpose`. A granted unusual-weather waiver means the EA
> has excused the sample — but it is recorded as a separate observation, and the sample's purpose is
> unchanged, so the archive's own compliance filter hands it back as a compliance sample. Every
> consumer must re-derive the exclusion, and each one that forgets assesses against samples the
> regulator has already set aside."*

The archive models the exclusion, and files it somewhere its own filters cannot act on. Four steps
below: the waiver exists as an observation; it and the results it excuses survive the compliance filter;
the sample's classification does not reflect it; so the rule falls to the consumer.


In [12]:
# Step 1. The waiver is an observation on the sample, beside the results it excuses.
import urllib.request

def ld(url):
    req = urllib.request.Request(url, headers={"Accept": "application/ld+json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

POINT, SAMPLE = "SW-50410119", "3160052"
BASE = f"https://environment.data.gov.uk/water-quality/sampling-point/{POINT}"

try:
    print("Every observation on sample", SAMPLE)
    rows, skip = [], 0
    while skip < 3000:                       # there is no per-sample route; page and filter
        page = ld(f"{BASE}/observation?skip={skip}&limit=250")["member"]
        if not page:
            break
        rows += [o for o in page if o["hasSample"]["id"].endswith(f"/{SAMPLE}")]
        skip += 250
    for o in sorted(rows, key=lambda o: o["observedProperty"]["notation"]):
        d = o["observedProperty"]
        print(f'   {d["notation"]:>6}  {d["prefLabel"][:40]:<40} = {o.get("hasSimpleResult")} {o.get("hasUnit", "")}')
except Exception as exc:                     # offline / endpoint unavailable
    print("live fetch unavailable:", exc)
    print("   0061  pH                                       = 7.54 PHUNITS")
    print("   0085  BOD : 5 Day ATU                          = 31 mg/l")
    print("   0111  Ammoniacal Nitrogen as N                 = 13.35 mg/l")
    print("   0135  Solids, Suspended at 105 C               = 39 mg/l")
    print("   4838  Unusual Weather Waiver (WRA)             = Granted Coded Result")


Every observation on sample 3160052


     0061  pH                                       = 7.54 PHUNITS
     0085  BOD : 5 Day ATU                          = 31 mg/l
     0111  Ammoniacal Nitrogen as N                 = 13.35 mg/l
     0135  Solids, Suspended at 105 C               = 39 mg/l
     4838  Unusual Weather Waiver (WRA)             = Granted Coded Result


### Step 2. The compliance filter keeps it — and keeps the results it excuses


In [13]:
PT = "SW-50951080"
try:
    for query in ("determinand=4838&complianceOnly=true", "determinand=4838"):
        doc_ = ld(f"https://environment.data.gov.uk/water-quality/sampling-point/{PT}"
                  f"/observation?{query}&limit=250")
        print(f"{query:<40} -> {doc_['totalItems']} observation(s)")
        for o in doc_["member"]:
            print(f"      {o['phenomenonTime'][:10]}  sample {o['hasSample']['id'].rsplit('/', 1)[-1]}"
                  f"  = {o.get('hasSimpleResult')}")
except Exception as exc:
    print("live fetch unavailable:", exc)
    print("determinand=4838&complianceOnly=true      -> 1 observation(s)")
    print("      2018-02-07  sample 3344474  = Granted")


determinand=4838&complianceOnly=true     -> 1 observation(s)
      2018-02-07  sample 3344474  = Granted


determinand=4838                         -> 1 observation(s)
      2018-02-07  sample 3344474  = Granted


`complianceOnly=true` does not drop the waiver, and — the part that matters — it does not drop the
**numeric results sitting on the same sample** either. Those results are in this project's committed
compliance cache right now.

### Step 3. The sample's own classification says nothing about it

`complianceOnly=true` is a filter over `samplingPurpose`. So what purpose do the waived samples carry?


In [14]:
waivers = pd.read_csv(ROOT / "notebooks" / "data" / "weather_waivers.csv", dtype=str)
comp["sample"] = comp.id.str.extract(r"/sample/(\d+)/")
on_waived = comp[comp["sample"].isin(set(waivers["sample"]))]

print("samplingPurpose of the observations on WAIVED samples, as returned by complianceOnly=true:\n")
print(on_waived.groupby(["sample", "samplingPurpose"]).size().rename("observations").to_string())
print("\nThe purpose vocabulary the filter selects on:\n")
print(comp.samplingPurpose.value_counts().rename("observations").to_string())
print("\nA waived sample is classified identically to one with no waiver. There is no")
print("'waived compliance' purpose for complianceOnly=true to exclude.")


samplingPurpose of the observations on WAIVED samples, as returned by complianceOnly=true:

sample   samplingPurpose                                       
3062097  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    3
3062120  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    1
3062243  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    3
3062245  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    4
3062265  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    3
3160052  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    4
3308281  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    3
3344474  WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    3

The purpose vocabulary the filter selects on:

samplingPurpose
COMPLIANCE AUDIT (PERMIT)                                 26739
WATER QUALITY OPERATOR SELF MONITORING COMPLIANCE DATA    14604
WATER QUALITY UWWTD MONITORING DATA                        2656
COMPLIANCE FORMAL (PERMIT)          

That is the finding. The waiver is a **decision about the sample's status** — the EA has set it aside
for assessment — but it is modelled orthogonally to the classification the API filters on. So the
archive simultaneously asserts *"this is a compliance sample"* (purpose) and *"this sample is excused"*
(a sibling observation), and its own compliance filter honours only the first.

Had `samplingPurpose` carried a waived variant, or had the waiver been an attribute of the sample rather
than an observation alongside it, `complianceOnly=true` would mean what its name says and every consumer
would get the exclusion for free.

### Step 4. So the business logic lands on the consumer — and this consumer dropped it


In [15]:
print("waiver rows in this project's committed compliance cache:",
      int((comp["determinand.notation"] == "4838").sum()))
print("compliance observations it holds that sit on a waived sample:", len(on_waived))
print()
print("Why: fetch() in ttl/breaches/fetch_compliance_observations.py asks per (sampling point,")
print("determinand), for the determinands a permit holds conditions for --")
print(f"   determinands ever requested: {comp['determinand.notation'].nunique()};  is 4838 among them?"
      f" {'4838' in set(comp['determinand.notation'])}")
print()
print("To apply the exclusion a consumer must, unprompted: know 4838 exists; request it as well as the")
print("determinands it cares about; join it to the results by sample; and read its coded value --")
print("because presence is not the answer.")
print()
print(waivers.groupby("result").size().rename("waivers").to_string())


waiver rows in this project's committed compliance cache: 0
compliance observations it holds that sit on a waived sample: 24

Why: fetch() in ttl/breaches/fetch_compliance_observations.py asks per (sampling point,
determinand), for the determinands a permit holds conditions for --
   determinands ever requested: 27;  is 4838 among them? False

To apply the exclusion a consumer must, unprompted: know 4838 exists; request it as well as the
determinands it cares about; join it to the results by sample; and read its coded value --
because presence is not the answer.

result
Granted        7
Not Granted    1


In [16]:
# What it costs in this build, and the sample that was NOT excused.
import pyoxigraph as ox
breaches = ox.Store()
breaches.bulk_load(path=str(ROOT / "ttl" / "breaches.ttl"), format=ox.RdfFormat.TURTLE)
judged = {str(r[0].value) for r in breaches.query(
    "SELECT ?o WHERE { ?b ?p ?o FILTER(CONTAINS(STR(?o), '/observation/')) }")}
print(f"observations underpinning a delivered breach:  {len(judged)}")
print(f"delivered breaches resting on a waived sample: {len(set(on_waived.id) & judged)}   "
      "-> this build's published assessments are unaffected; the defect is latent")
print()
not_granted = set(waivers[waivers.result == "Not Granted"]["sample"])
print("The refused waiver -- these results stand, and a consumer keying on presence would wrongly drop them:")
display(on_waived[on_waived["sample"].isin(not_granted)][
    ["samplingPoint.notation", "phenomenonTime", "determinand.notation", "result", "unit"]])


observations underpinning a delivered breach:  290
delivered breaches resting on a waived sample: 0   -> this build's published assessments are unaffected; the defect is latent

The refused waiver -- these results stand, and a consumer keying on presence would wrongly drop them:


,samplingPoint.notation,phenomenonTime,determinand.notation,result,unit
16399,SW-50550374,2016-11-21T09:24:00,0085,21,MILLIGRAM PER LITRE
16711,SW-50550374,2016-11-21T09:24:00,0111,6.65,MILLIGRAM PER LITRE
17023,SW-50550374,2016-11-21T09:24:00,0135,23,MILLIGRAM PER LITRE


### Every waiver in the catchment

Swept per sampling point, because there is no catchment-level route to them. Regenerate with
`python notebooks/sweep_waivers.py`.


In [17]:
print(f"sampling points swept: 161   waiver observations found: {len(waivers)}\n")
display(waivers.sort_values("date").reset_index(drop=True))


sampling points swept: 161   waiver observations found: 8



,sampling_point,determinand,label,date,sample,result
0,SW-50530190,4838,Unusual Weather Waiver (WRA),2010-12-21,3062243,Granted
1,SW-50540186,4838,Unusual Weather Waiver (WRA),2010-12-21,3062245,Granted
2,SW-50550374,4838,Unusual Weather Waiver (WRA),2010-12-21,3062097,Granted
3,SW-50440001,4838,Unusual Weather Waiver (WRA),2010-12-23,3062265,Granted
4,SW-50580185,4838,Unusual Weather Waiver (WRA),2010-12-29,3062120,Granted
5,SW-50410119,4838,Unusual Weather Waiver (WRA),2013-03-14,3160052,Granted
6,SW-50550374,4838,Unusual Weather Waiver (WRA),2016-11-21,3308281,Not Granted
7,SW-50951080,4838,Unusual Weather Waiver (WRA),2018-02-07,3344474,Granted


**The shape of the defect.** The waiver reaches the consumer, and it arrives *next to* rather than
*attached to* the thing it modifies. An observation cannot change how the API classifies the sample it
sits on, so `complianceOnly=true` cannot act on it, and every consumer re-implements the rule or silently
skips it. Attaching the decision to the sample — a waived variant of `samplingPurpose`, or an attribute
of the sample itself — would make the exclusion free for everyone who asks for compliance data.


---
## C.2.5 The JSON-LD surface: unresolvable concepts and inconsistent scheme

> *"Sampling-point type and status come back as blank-node concepts with no resolvable IRI … Sampling
> point IRIs are returned as `https:` where they are `http:` elsewhere."*

First, the scheme inconsistency — visible without any network access, between two files the archive
itself produced:


In [18]:
print("bulk download, observation id :", bulk.id.iloc[0])
print("compliance API, observation id:", comp.id.iloc[0])
print()
print(f"bulk rows using https:                  {bulk.id.str.startswith('https:').mean():.0%}")
print(f"compliance rows using http (not https): "
      f"{(comp.id.str.startswith('http:') & ~comp.id.str.startswith('https:')).mean():.0%}")
print()
print("Same archive, same sampling points, two IRI schemes. In RDF these are two different")
print("resources, so a join across the two surfaces returns nothing unless one is normalised.")


bulk download, observation id : https://environment.data.gov.uk/water-quality/sampling-point/SW-50410119/sample/3444071/observation/0061
compliance API, observation id: http://environment.data.gov.uk/water-quality/sampling-point/SW-50410119/sample/2882195/observation/0061

bulk rows using https:                  100%
compliance rows using http (not https): 100%

Same archive, same sampling points, two IRI schemes. In RDF these are two different
resources, so a join across the two surfaces returns nothing unless one is normalised.


In [19]:
# Live: what a sampling point's own JSON-LD says about its type and status.
import urllib.request

req = urllib.request.Request(
    "http://environment.data.gov.uk/water-quality/sampling-point/SW-50950709",
    headers={"Accept": "application/ld+json"})
try:
    with urllib.request.urlopen(req, timeout=30) as r:
        doc = json.load(r)
        print("requested over http:, served from:", r.url, "\n")
    node = doc["member"][0]
    for key in ("samplingPointType", "samplingPointStatus"):
        print(f"{key}:")
        print(json.dumps(node[key], indent=2))
        print("   resolvable identifier?",
              "NO -- '@id' is a blank node" if str(node[key].get("@id", "")).startswith("_:") else "yes")
        print()
except Exception as exc:                     # offline / endpoint unavailable
    print("live fetch unavailable:", exc)
    print("\nWhat the committed extract shows instead -- the type carries a notation and label,")
    print("but no IRI, so an identifier has to be minted locally:")
    print(pd.read_csv(ROOT / "ttl" / "regulation" / "sampling_points.csv", dtype=str)[
        ["sp_notation","type_notation","type_label","status_label"]].head(5).to_string(index=False))


requested over http:, served from: https://environment.data.gov.uk/water-quality/sampling-point/SW-50950709 

samplingPointType:
{
  "@id": "_:samplingPointType#SA",
  "@type": [
    "skos:Concept",
    "sosa:Property"
  ],
  "prefLabel": "SEWAGE DISCHARGES - FINAL/TREATED EFFLUENT - WATER COMPANY",
  "notation": "SA"
}
   resolvable identifier? NO -- '@id' is a blank node

samplingPointStatus:
{
  "@id": "_:status#O",
  "@type": [
    "skos:Concept",
    "sosa:Property"
  ],
  "prefLabel": "OPEN",
  "notation": "O"
}
   resolvable identifier? NO -- '@id' is a blank node



The type and status are `skos:Concept`s with **blank-node identifiers** (`_:samplingPointType#SA`).
A blank node is scoped to the document it appeared in, so nothing outside can refer to it — two
consumers fetching the same point get two unrelated identifiers for the same concept. A consumer has to
mint its own IRI from the notation, which is the kind of local invention linked data exists to avoid.

Note the redirect printed above too: the request went to `http:` and was served from `https:`, while the
compliance API publishes `http:` identifiers. The archive is inconsistent with itself.


In [20]:
sp_ref = pd.read_csv(ROOT / "ttl" / "regulation" / "sampling_points.csv", dtype=str)
print("The sampling-point types the catchment uses, each needing a locally minted identifier:")
sp_ref[["type_notation","type_label"]].drop_duplicates().sort_values("type_notation").reset_index(drop=True)


The sampling-point types the catchment uses, each needing a locally minted identifier:


,type_notation,type_label
0,AC,AGRICULTURE - FISH FARMING - WATER COMPANY
1,AD,AGRICULTURE - WATER CRESS FARMING
2,AF,AGRICULTURE - FISH FARMING - NOT WATER COMPANY
3,BA,GROUNDWATER - BOREHOLE
4,BB,GROUNDWATER - SPRING
5,CA,SALINE WATER - DESIGNATED BATHING BEACHES
6,CE,SALINE WATER - ESTUARINE SITES - NON BATHING/SHELLFISH
7,F6,FRESHWATER - RIVERS
8,FA,FRESHWATER - LAKES/PONDS/RESERVOIRS
9,FC,FRESHWATER - COMPARATIVE INLET POINTS
